# 5.5.6 — Unidades de Conservação: análises, tabelas e mapas

Mantém a base expandida da **versão 2**, sem excluir repetições de REF_BACEN ou id_gleba2. Reproduz os totais e as análises por **grupo de proteção, categoria de manejo, esfera administrativa, ano e categoria de produtor**. Acrescenta:

1. Tabela completa com uma linha por UC, além das 20 UCs com mais registros.
2. Mapa do Cerrado com contornos fortes das UCs, glebas e interseções.
3. Mapa da UF com **mais registros**, com limites estaduais e ampliação da UC de maior frequência nessa UF.
4. Mapas das **duas UCs com mais registros**, cada um com visão completa e ampliação de uma concentração local.
5. Gráficos por grupo de proteção e por ano; conferências, textos e ZIP para a reescrita.

**Como executar:** no Google Colab, use **Ambiente de execução → Executar tudo** e autorize a montagem do Drive. Os caminhos foram conferidos no Drive em 18/09/2026. A origem tem aproximadamente **12,74 GB**: a primeira leitura é sequencial e pode demorar; um cache reduzido evita repetir essa leitura nas próximas execuções. Não é necessário carregar o CSV inteiro na memória.

O resumo V2 aponta **Mato Grosso do Sul** como a UF com mais registros (3.753) e as APAs **Ilha do Bananal/Cantão** (2.804; ID 8) e **Serra da Ibiapaba** (1.720; ID 558) como as duas primeiras UCs. O notebook recalcula os rankings; não força esses resultados.

**Interpretação:** as classes seguem os atributos do CNUC preservados na V2. Não se infere irregularidade, domínio, zoneamento ou situação jurídica na data da operação a partir da interseção. Valores financeiros somados nas linhas podem se repetir e não são rateados pela área. A união geral é calculada diretamente: não se somam as uniões dos grupos ou das UCs.

Os resultados serão salvos em `complementos_unidades_conservacao_5_5_6/<data_hora>`. Compartilhe essa pasta ou o ZIP final para a reescrita do Word.

## 1. Preparar o Colab e montar o Drive

In [1]:
%pip -q install "geopandas>=1.0,<2" "shapely>=2.0,<3" "pyogrio>=0.9" "pyproj>=3.6" openpyxl requests tqdm
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Caminhos e parâmetros
A entrada é `base_registros_espaciais_sem_deduplicacao.csv`, com 18.617 linhas na V2 conferida. A base de partes municipais não substitui essa entrada. Os arquivos de resumo são usados somente como referências de conferência. Os vetores do IBGE são reaproveitados quando já existem no cache dos complementos anteriores; caso contrário, são baixados das URLs oficiais configuradas.

In [2]:
from pathlib import Path
from datetime import datetime, timezone
RAIZ = Path('/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios')
PASTA_V2 = RAIZ / 'intersecoes' / 'unidades de conservacao' / 'resultados_por_id_gleba2' / 'versao_2'
ARQUIVO_BASE = PASTA_V2 / 'bases_analiticas' / 'base_registros_espaciais_sem_deduplicacao.csv'
ID_DRIVE_BASE = '1rL27sydppHpoHKOWcxinr1dGrtBlaxpY'
ARQUIVO_UFS = None       # opcional: caminho de ZIP, SHP, GPKG ou GeoJSON
ARQUIVO_BIOMAS = None    # opcional: caminho de ZIP, SHP, GPKG ou GeoJSON
FONTE_UFS = 'IBGE, Malha de Unidades da Federação, 2024'
FONTE_BIOMAS = 'IBGE, Biomas do Brasil, 2019, escala 1:250.000'
URL_UFS = 'https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_municipais/municipio_2024/Brasil/BR_UF_2024.zip'
URL_BIOMAS = 'https://geoftp.ibge.gov.br/informacoes_ambientais/estudos_ambientais/biomas/vetores/Biomas_250mil.zip'
COLUNA_UF = 'SIGLA_UF'
COLUNA_BIOMA = 'Bioma'
CRS_WKT = 'EPSG:4674'
CRS_ANALISE = 'EPSG:5880'
ANOS = [2021, 2022, 2023, 2024, 2025]
NUMERO_TERRITORIOS_DETALHE = 2
DPI = 400
FORMATOS = ['png', 'pdf', 'svg']
MOSTRAR_FIGURAS = True
BAIXAR_ZIP = True
TOLERANCIA_AREA_HA = 0.05
REFERENCIA = dict(registros=18617,glebas=14621,territorios=208,refs=11757,
    area_acumulada_ha=1494554.545179507,area_uniao_ha=790318.9874052661,
    soma_valores_linhas=21606212830.34)
# Descoberta automática da coluna com o polígono completo da UC.
# Preencha somente se o notebook informar mais de uma candidata.
COLUNA_GEOMETRIA_UC = None
SIMPLIFICACAO_GERAL_M = 50
SIMPLIFICACAO_ESTADO_M = 10
RECRIAR_CACHE = False
VERSAO_CACHE = 'uc556_v1'
EXECUCAO = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
SAIDA = RAIZ / 'complementos_unidades_conservacao_5_5_6' / EXECUCAO
CACHE = RAIZ / 'complementos_unidades_conservacao_5_5_6' / '_cache'
for p in [SAIDA,CACHE,SAIDA/'tabelas',SAIDA/'mapas']:p.mkdir(parents=True,exist_ok=True)
print('Base:',ARQUIVO_BASE)
print('Resultados:',SAIDA)

ARQUIVO_RESUMO_UNIDADES = PASTA_V2 / 'tabelas' / '04_resumo_por_unidade_territorial.csv'

ARQUIVO_RESUMO_PRODUTOR = PASTA_V2 / 'tabelas' / '11_resumo_por_tipo_produtor.csv'

ARQUIVO_RESUMO_UFS = PASTA_V2 / 'tabelas' / '08_resumo_por_uf_espacial.csv'
ARQUIVO_RESUMO_ANOS = PASTA_V2 / 'tabelas' / '10_resumo_por_ano.csv'
ARQUIVO_RESUMO_GRUPOS = PASTA_V2 / 'tabelas' / '06_resumo_por_tipologia_2.csv'
ARQUIVO_RESUMO_CATEGORIAS = PASTA_V2 / 'tabelas' / '05_resumo_por_tipologia_1.csv'
ARQUIVO_RESUMO_ESFERAS = PASTA_V2 / 'tabelas' / '07_resumo_por_tipologia_3.csv'
(SAIDA/'graficos').mkdir(exist_ok=True)


Base: /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/intersecoes/unidades de conservacao/resultados_por_id_gleba2/versao_2/bases_analiticas/base_registros_espaciais_sem_deduplicacao.csv
Resultados: /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/complementos_unidades_conservacao_5_5_6/20260918_175505_991051


## 3. Contexto cartográfico
Limites de UFs e bioma são obtidos do IBGE. São preservados os polígonos das UCs da versão 2, evitando misturar versões territoriais. O arquivo grande é identificado por caminho, tamanho e data de modificação; não é relido apenas para calcular um hash integral.

In [3]:
import json, hashlib, zipfile, re, math, platform, textwrap
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
from shapely.geometry import box
from pyproj import CRS, Transformer, Geod
import requests
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
from IPython.display import display
ORIGENS={}

def hash_arquivo(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()

def entrada_drive():
    if not ARQUIVO_BASE.is_file():
        raise FileNotFoundError('Base não encontrada no Drive montado: '+str(ARQUIVO_BASE)+
            '\nConfira a conta e o caminho. ID do arquivo conferido: '+ID_DRIVE_BASE)
    return ARQUIVO_BASE

def obter_vetor(local,url,nome):
    if local:
        p=Path(local)
        if not p.is_file():raise FileNotFoundError(p)
        return p
    # Reutiliza vetores já baixados nos complementos anteriores, sem alterar suas fontes.
    for pasta in sorted(RAIZ.glob('complementos_*')):
        for sub in ('_cache_contexto','_cache'):
            anterior=pasta/sub/nome
            if anterior.is_file() and zipfile.is_zipfile(anterior):return anterior
    p=CACHE/nome
    if not (p.is_file() and zipfile.is_zipfile(p)):
        tmp=p.with_suffix('.part')
        with requests.get(url,stream=True,timeout=(30,180)) as r:
            r.raise_for_status()
            with tmp.open('wb') as f:
                for b in r.iter_content(1024*1024):f.write(b)
        if not zipfile.is_zipfile(tmp):raise ValueError('O download não retornou ZIP: '+url)
        tmp.replace(p)
    return p

def ler_vetor(p):
    z=gpd.read_file('zip://'+str(p) if str(p).lower().endswith('.zip') else p)
    if z.crs is None:raise ValueError('Vetor sem referência espacial: '+str(p))
    return z

CAMINHO=entrada_drive()
pufs=obter_vetor(ARQUIVO_UFS,URL_UFS,'BR_UF_2024.zip')
pbiomas=obter_vetor(ARQUIVO_BIOMAS,URL_BIOMAS,'Biomas_250mil_2019.zip')
for nome,p in [('base',CAMINHO),('ufs',pufs),('biomas',pbiomas)]:
    ORIGENS[nome]={'arquivo':str(p),'bytes':p.stat().st_size,'mtime_ns':p.stat().st_mtime_ns}
    if nome!='base':ORIGENS[nome]['sha256']=hash_arquivo(p)
ufs=ler_vetor(pufs);biomas=ler_vetor(pbiomas)
if COLUNA_UF not in ufs or COLUNA_BIOMA not in biomas:raise ValueError('Confira as colunas dos arquivos de contexto.')
ufs=ufs[[COLUNA_UF,'geometry']].rename(columns={COLUNA_UF:'UF'}).to_crs(CRS_ANALISE)
ufs['UF']=ufs.UF.astype(str).str.strip()
if ufs.UF.duplicated().any():ufs=ufs.dissolve(by='UF',as_index=False)
cer=biomas.loc[biomas[COLUNA_BIOMA].astype(str).str.strip().str.casefold().eq('cerrado'),['geometry']].to_crs(CRS_ANALISE)
if cer.empty:raise ValueError('Cerrado não encontrado no vetor de biomas.')
# Reparos apenas nos limites de contexto; qualquer alteração fica registrada.
ORIGENS['reparos_contexto']={}
for nome,z in [('ufs',ufs),('cerrado',cer)]:
    if z.geometry.isna().any() or z.geometry.is_empty.any():raise ValueError('Geometria ausente no contexto '+nome)
    mask=~z.is_valid;ORIGENS['reparos_contexto'][nome]=int(mask.sum())
    if mask.any():z.loc[mask,'geometry']=z.loc[mask,'geometry'].make_valid()
    if not z.geom_type.isin(['Polygon','MultiPolygon']).all():raise ValueError('Contexto não poligonal: '+nome)
cerrado=cer.geometry.union_all()
print('Entradas localizadas. Base:',round(CAMINHO.stat().st_size/1e9,2),'GB; leitura sequencial na próxima célula.')


Entradas localizadas. Base: 12.74 GB; leitura sequencial na próxima célula.


## 4. Leitura sequencial e cache
Cada UC e cada gleba têm uma cópia de sua geometria no cache; todas as linhas de interseção são preservadas. Identificadores são textuais. A coluna do limite **completo da UC** é identificada no cabeçalho e nunca substituída pela geometria da interseção.

Ausências, divergências entre cópias e problemas de validade são diagnosticados. Reparos após reprojeção são permitidos apenas em geometrias destinadas ao desenho e ficam registrados, inclusive quando o cache é reutilizado. As interseções dos cálculos permanecem sob validação rigorosa. O arquivo de origem não é alterado.

In [4]:
import csv as csvmod, sqlite3, zlib, tempfile, shutil, time
from shapely.ops import transform as transformar
from tqdm.auto import tqdm
csvmod.field_size_limit(2**31-1)

def numero_pt(v):
    s=str(v).strip()
    if s.lower() in ['', 'nan', 'none', 'null', '<na>']:return np.nan
    if ',' in s:s=s.replace('.','').replace(',','.')
    try:
        a=float(s)
        if not np.isfinite(a):raise ValueError(s)
        return a
    except ValueError:raise ValueError('Número não interpretável: '+s)

def csv(t,n):
    t.to_csv(SAIDA/'tabelas'/n,index=False,sep=';',decimal=',',encoding='utf-8-sig',float_format='%.9f')

with CAMINHO.open('r',encoding='utf-8-sig',newline='') as f:
    cabecalho=next(csvmod.reader(f,delimiter=';'))
if len(cabecalho)!=len(set(cabecalho)):raise ValueError('Há nomes de colunas duplicados no CSV.')
C={'gleba':'ID_GLEBA2','ref':'REF_BACEN','ano':'ANO_BASE','unidade':'UC_ID_INTERNO',
   'nome':'UC_NOME_UC','grupo':'UC_GRUPO','uf_territorio':'UC_UF',
   'categoria_uc':'UC_CATEGORIA','esfera':'UC_ESFERA','produtor_original':'TIPO_PRODUTOR_ANALISE','valor':'VL_PARC_CREDITO',
   'area_csv':'AREA_INTERSECAO_HA','inter':'GEOMETRIA_INTERSECAO_WKT','geom_gleba':'GT_GEOMETRIA'}
# Campos administrativos conferidos no arquivo 01_campos_identificados.csv da V2.
# A geometria territorial completa é descoberta no cabeçalho sem carregar os 12,74 GB.
if COLUNA_GEOMETRIA_UC:
    C['geom_territorio']=COLUNA_GEOMETRIA_UC
else:
    conhecidas=['GEOMETRIA_UC_WKT','GEOMETRIA_UCS_WKT','GEOMETRIA_UNIDADE_CONSERVACAO_WKT',
                'GEOMETRIA_UNIDADE_DE_CONSERVACAO_WKT','UC_GEOMETRIA_WKT','UC_GEOMETRIA']
    candidatas=[c for c in cabecalho if c in conhecidas or
        ('GEOMETR' in c.upper() and any(t in c.upper() for t in ('CONSERVAC','UC_','_UC')))]
    candidatas=list(dict.fromkeys(candidatas))
    if len(candidatas)!=1:
        (SAIDA/'cabecalho_base.txt').write_text('\n'.join(cabecalho),encoding='utf-8')
        raise ValueError('Não foi possível selecionar uma única geometria COMPLETA da UC. '
            'Candidatas: '+str(candidatas)+'. Veja cabecalho_base.txt e preencha COLUNA_GEOMETRIA_UC. '
            'Não use GEOMETRIA_INTERSECAO_WKT em seu lugar.')
    C['geom_territorio']=candidatas[0]
C['linha']='_ID_LINHA_ORIGINAL' if '_ID_LINHA_ORIGINAL' in cabecalho else None
faltam=set(v for v in C.values() if v)-set(cabecalho)
if faltam:raise ValueError('Colunas obrigatórias ausentes: '+str(sorted(faltam)))
if C['geom_territorio'] in [C['inter'],C['geom_gleba']]:raise ValueError('Coluna da UC não pode ser a interseção ou a gleba.')
ORIGENS['identificador_linha']=C['linha'] or 'posição sequencial no CSV de registros espaciais'
assinatura={'origem':ORIGENS['base'],'colunas':C,'anos':ANOS,'crs_wkt':CRS_WKT,
            'crs_analise':CRS_ANALISE,'versao':VERSAO_CACHE}
chave=hashlib.sha256(json.dumps(assinatura,sort_keys=True).encode()).hexdigest()[:20]
BANCO=CACHE/f'ucs_reduzido_{chave}.sqlite'
converter=Transformer.from_crs(CRS_WKT,CRS_ANALISE,always_xy=True).transform

REPAROS_CARTOGRAFICOS=[]

def poligono(wkt,identificacao):
    if not str(wkt).strip():raise ValueError('Geometria ausente: '+identificacao)
    g=shapely.force_2d(shapely.from_wkt(wkt,on_invalid='raise'))
    if g is None or g.is_empty or g.geom_type not in ('Polygon','MultiPolygon'):
        raise ValueError('Geometria vazia ou não poligonal: '+identificacao)
    if not g.is_valid:
        raise ValueError('Geometria inválida na origem: '+identificacao+'; '+shapely.is_valid_reason(g))
    a,b,c,d=g.bounds
    if not np.isfinite([a,b,c,d]).all():raise ValueError('Coordenadas não finitas: '+identificacao)
    if CRS.from_user_input(CRS_WKT).is_geographic and not (-80<a<-30 and -40<b<10 and -80<c<-30 and -40<d<10):
        raise ValueError('Coordenadas incompatíveis com WKT geográfico do Brasil: '+identificacao)
    g=transformar(converter,g)
    if not np.isfinite(g.bounds).all() or not np.isfinite(g.area):
        raise ValueError('Reprojeção não finita: '+identificacao+'. Confira os CRS.')
    if not g.is_valid:
        motivo=shapely.is_valid_reason(g)
        if not identificacao.startswith(('territorio=','gleba=')):
            raise ValueError('Interseção inválida após reprojeção: '+identificacao+'; '+motivo)
        antes=float(g.area);pilha=[shapely.make_valid(g)];partes=[];descartados=[]
        while pilha:
            parte=pilha.pop()
            if parte.is_empty:continue
            if parte.geom_type=='Polygon':partes.append(parte)
            elif parte.geom_type in ('MultiPolygon','GeometryCollection'):pilha.extend(parte.geoms)
            else:descartados.append(parte.geom_type)
        if not partes:raise ValueError('Reparo sem componente poligonal: '+identificacao)
        g=shapely.union_all(partes)
        if g.is_empty or not g.is_valid or g.geom_type not in ('Polygon','MultiPolygon') or not np.isfinite(g.area) or g.area<=0:
            raise ValueError('Reparo cartográfico não recuperável: '+identificacao)
        REPAROS_CARTOGRAFICOS.append({'identificacao':identificacao,'motivo':motivo,
            'area_antes_ha':antes/10000,'area_depois_ha':float(g.area)/10000,
            'diferenca_area_ha':(float(g.area)-antes)/10000,'componentes_descartados':','.join(descartados)})
        print('Reparo de desenho documentado:',identificacao)
    if g.is_empty or g.area<=0:raise ValueError('Geometria sem área positiva: '+identificacao)
    return g

def compactar(g):return sqlite3.Binary(zlib.compress(shapely.to_wkb(g),level=1))
def descompactar(v):return shapely.from_wkb(zlib.decompress(v))

def cache_pronto(p):
    if not p.is_file():return False
    try:
        with sqlite3.connect(p) as db:
            r=db.execute('SELECT valor FROM meta WHERE chave="assinatura"').fetchone()
            ok=db.execute('SELECT valor FROM meta WHERE chave="completo"').fetchone()
            return r is not None and json.loads(r[0])==assinatura and ok is not None and ok[0]=='sim'
    except (sqlite3.Error,ValueError):return False

def construir_cache(origem,destino):
    REPAROS_CARTOGRAFICOS.clear()
    vistos={};atributos_vistos={};linhas_vistas=set();variantes=0
    inicio=time.monotonic();numero=0;linha_id=None;unidade=None;gleba=None
    # Banco intermediário em disco local: evita milhares de pequenas escritas no Drive.
    with tempfile.TemporaryDirectory(prefix='ucs_553_') as td:
        temporario=Path(td)/'cache.sqlite'
        db=sqlite3.connect(temporario)
        try:
            db.executescript('''
                CREATE TABLE geometrias (tipo TEXT, id TEXT, atributos TEXT, wkb BLOB, PRIMARY KEY(tipo,id));
                CREATE TABLE registros (numero INTEGER PRIMARY KEY, atributos TEXT, wkb BLOB);
                CREATE TABLE meta (chave TEXT PRIMARY KEY, valor TEXT);
            ''')
            def guardar_unica(tipo,ident,wkt,attrs):
                nonlocal variantes
                key=(tipo,ident);h=hashlib.sha256(wkt.encode('utf-8')).hexdigest()
                attrs_str=json.dumps(attrs,sort_keys=True,ensure_ascii=False)
                if key in vistos:
                    if atributos_vistos[key]!=attrs_str:raise ValueError('Atributos divergentes para '+tipo+'='+ident)
                    if h in vistos[key]:return
                    original=descompactar(db.execute('SELECT wkb FROM geometrias WHERE tipo=? AND id=?',key).fetchone()[0])
                    if not original.equals(poligono(wkt,tipo+'='+ident)):
                        raise ValueError('Cópias geométricas divergentes para '+tipo+'='+ident)
                    vistos[key].add(h);variantes+=1;return
                g=poligono(wkt,tipo+'='+ident)
                db.execute('INSERT INTO geometrias VALUES (?,?,?,?)',(tipo,ident,attrs_str,compactar(g)))
                vistos[key]={h};atributos_vistos[key]=attrs_str
            with origem.open('r',encoding='utf-8-sig',newline='') as f:
                reader=csvmod.DictReader(f,delimiter=';')
                for numero,row in enumerate(tqdm(reader,total=REFERENCIA['registros'],desc='Leitura sequencial',unit='linha'),1):
                    if None in row or any(v is None for v in row.values()):raise ValueError('Quantidade de campos diferente do cabeçalho.')
                    attrs={k:str(row[v]).strip() for k,v in C.items() if v and k not in ('inter','geom_gleba','geom_territorio','valor','area_csv')}
                    linha_id=attrs.get('linha',str(numero));attrs['linha']=linha_id
                    for k in ['linha','unidade','gleba','ref','ano']:
                        if attrs[k].lower() in ('','nan','none','null','<na>'):raise ValueError('Identificador ausente: '+k)
                    unidade=attrs['unidade'];gleba=attrs['gleba']
                    if linha_id in linhas_vistas:raise ValueError('ID de linha repetido; confira se a origem é a base de partes municipais.')
                    linhas_vistas.add(linha_id)
                    ano=numero_pt(attrs['ano'])
                    if ano not in ANOS:raise ValueError('Ano fora de 2021–2025: '+attrs['ano'])
                    attrs['ano']=str(int(ano));attrs['codigo_territorio']=unidade
                    for k in ['nome','grupo','uf_territorio','categoria_uc','esfera','produtor_original']:
                        if attrs[k].lower() in ('','nan','none','null','<na>'):
                            attrs[k]='Sem denominação — ID '+unidade if k=='nome' else 'Não informado'
                    for campo_epsg in ['UC_EPSG']:
                        if campo_epsg in row and str(row[campo_epsg]).strip():
                            if numero_pt(row[campo_epsg])!=CRS.from_user_input(CRS_WKT).to_epsg():
                                raise ValueError(campo_epsg+' diverge do CRS_WKT configurado.')
                    attrs['valor']=numero_pt(row[C['valor']]);attrs['area_csv']=numero_pt(row[C['area_csv']])
                    if not np.isfinite(attrs['area_csv']):raise ValueError('Área de origem ausente.')
                    attrs_territorio={k:attrs[k] for k in ['unidade','codigo_territorio','nome','grupo','uf_territorio','categoria_uc','esfera']}
                    guardar_unica('territorio',unidade,row[C['geom_territorio']],attrs_territorio)
                    guardar_unica('gleba',gleba,row[C['geom_gleba']],{'gleba':gleba})
                    g=poligono(row[C['inter']],'interseção, linha '+linha_id)
                    attrs['area_ha']=g.area/10000
                    db.execute('INSERT INTO registros VALUES (?,?,?)',(numero,json.dumps(attrs,ensure_ascii=False),compactar(g)))
                    if numero%500==0:db.commit()
            if numero==0:raise ValueError('Base vazia.')
            info={'linhas':numero,'variantes_wkt_equivalentes':variantes,'duracao_segundos':time.monotonic()-inicio,'reparos_cartograficos':list(REPAROS_CARTOGRAFICOS)}
            for k,v in [('assinatura',json.dumps(assinatura)),('completo','sim'),('leitura',json.dumps(info))]:
                db.execute('INSERT INTO meta VALUES (?,?)',(k,v))
            db.commit()
            if db.execute('PRAGMA quick_check').fetchone()[0]!='ok':raise RuntimeError('Falha na integridade do cache.')
        except Exception as exc:
            diagnostico={'linha_sequencial':numero,'linha_id':linha_id,'unidade':unidade,'gleba':gleba,'erro':str(exc)}
            (SAIDA/'diagnostico_leitura.json').write_text(json.dumps(diagnostico,ensure_ascii=False,indent=2),encoding='utf-8')
            raise
        finally:db.close()
        incompleto=destino.with_suffix('.part')
        shutil.copyfile(temporario,incompleto)
        if incompleto.stat().st_size!=temporario.stat().st_size:raise RuntimeError('Cópia incompleta do cache.')
        incompleto.replace(destino)
    print('Cache reduzido salvo:',destino,'—',round(destino.stat().st_size/1e6,1),'MB')

if RECRIAR_CACHE or not cache_pronto(BANCO):construir_cache(CAMINHO,BANCO)
else:print('Reutilizando cache concluído e compatível:',BANCO.name)

def carregar_geometrias(tipo,ids=None,simplificar=0):
    escolhidos=None if ids is None else set(map(str,ids));result=[]
    with sqlite3.connect(BANCO) as db:
        # Filtra por ID antes de descompactar geometrias não solicitadas.
        for ident,attrs,wkb in db.execute('SELECT id,atributos,wkb FROM geometrias WHERE tipo=?',(tipo,)):
            if escolhidos is not None and ident not in escolhidos:continue
            g=descompactar(wkb)
            if simplificar:g=g.simplify(simplificar,preserve_topology=True)
            result.append({**json.loads(attrs),'geometry':g})
    if not result:return gpd.GeoDataFrame({'geometry':[]},crs=CRS_ANALISE)
    return gpd.GeoDataFrame(result,geometry='geometry',crs=CRS_ANALISE)

reg=[]
with sqlite3.connect(BANCO) as db:
    for attrs,wkb in db.execute('SELECT atributos,wkb FROM registros ORDER BY numero'):
        reg.append({**json.loads(attrs),'geometry':descompactar(wkb)})
    ORIGENS['leitura_cache']=json.loads(db.execute('SELECT valor FROM meta WHERE chave="leitura"').fetchone()[0])
x=gpd.GeoDataFrame(reg,geometry='geometry',crs=CRS_ANALISE);del reg
# Estas duas coleções servem apenas ao desenho de visão geral; os originais estão no cache.
territorios=carregar_geometrias('territorio',simplificar=SIMPLIFICACAO_GERAL_M)
glebas=carregar_geometrias('gleba',simplificar=SIMPLIFICACAO_GERAL_M)
area_check=x[['linha','unidade','gleba','area_csv','area_ha']].copy()
area_check['diferenca_ha']=area_check.area_ha-area_check.area_csv
csv(area_check,'conferencia_areas_por_linha.csv')
if area_check.diferenca_ha.abs().max()>TOLERANCIA_AREA_HA:
    raise ValueError('Área calculada diverge da V2 em alguma linha. Confira CRS e conferencia_areas_por_linha.csv.')
auditoria_nomes=territorios.drop(columns='geometry').copy()
csv(auditoria_nomes,'identificacao_ucs.csv')
print(f'{len(x):,} linhas preservadas; {len(glebas):,} glebas; {len(territorios):,} UCs.'.replace(',','.'))
print('Coluna de limite completo:',C['geom_territorio'])

# Exporta os reparos da execução de origem do cache, inclusive quando ele é reutilizado.
reparos=pd.DataFrame(ORIGENS['leitura_cache'].get('reparos_cartograficos',[]),columns=[
 'identificacao','motivo','area_antes_ha','area_depois_ha','diferenca_area_ha','componentes_descartados'])
csv(reparos,'reparos_geometrias_desenho.csv')
(SAIDA/'reparos_geometrias_desenho.json').write_text(reparos.to_json(orient='records',force_ascii=False,indent=2),encoding='utf-8')
print('Reparos cartográficos documentados:',len(reparos))


Leitura sequencial:   0%|          | 0/18617 [00:00<?, ?linha/s]

ValueError: Geometria vazia ou não poligonal: territorio=607

## 5. Totais por UC e preservação das análises anteriores
Os indicadores são recalculados diretamente nos registros, com união geométrica por agrupamento. A tabela completa usa `UC_ID_INTERNO`, sem fundir nomes semelhantes. Os grupos, categorias de manejo e esferas administrativas são preservados literalmente; variações e ausências permanecem visíveis.

São conferidos os totais globais, os resultados por UC, por grupo, por categoria, por esfera, por ano e por produtor contra os resumos V2 disponíveis. Divergências são exportadas, sem ajuste artificial dos resultados. O cálculo da área compartilhada entre os grupos explica a não aditividade de suas uniões.

In [ ]:
def metricas(z):
    return dict(registros=len(z),glebas=z.gleba.nunique(),refs=z.ref.nunique(),
        area_acumulada_ha=float(z.area_ha.sum()),area_uniao_ha=float(z.geometry.union_all().area/10000),
        soma_valores_linhas=float(z.valor.sum(min_count=1)),linhas_sem_valor=int(z.valor.isna().sum()))
res=[]
for unidade,z in tqdm(x.groupby('unidade',sort=False),total=x.unidade.nunique(),desc='Totais por UC'):
    t=territorios.loc[territorios.unidade.eq(unidade)].iloc[0]
    res.append({'unidade':unidade,'codigo_territorio':t.codigo_territorio,'nome':t['nome'],'grupo':t.grupo,'categoria_uc':t.categoria_uc,'esfera':t.esfera,'uf_territorio':t.uf_territorio,**metricas(z)})
por_territorio=pd.DataFrame(res).sort_values(['registros','nome','unidade'],ascending=[False,True,True]).reset_index(drop=True)
por_territorio.insert(0,'posicao',np.arange(1,len(por_territorio)+1))
geral={**metricas(x),'territorios':len(territorios)}
controle=[]
for k,anterior in REFERENCIA.items():
    valor=geral[k];tol=TOLERANCIA_AREA_HA if k.startswith('area') else (0.02 if k=='soma_valores_linhas' else 0)
    controle.append({'indicador':k,'calculado':valor,'referencia_v2':anterior,'diferenca':valor-anterior,'confere':abs(valor-anterior)<=tol})
conferencia=pd.DataFrame(controle)
csv(conferencia,'conferencia_com_texto_5_5_6.csv')
if not conferencia.confere.all():
    print('ATENÇÃO: há divergências com a versão anterior; os produtos usarão os resultados calculados, sem forçar os totais.')
assert por_territorio.registros.sum()==len(x)
assert np.isclose(por_territorio.area_acumulada_ha.sum(),geral['area_acumulada_ha'])
csv(por_territorio,'totais_por_unidade_conservacao_completo.csv')
csv(pd.DataFrame([geral]),'totais_gerais.csv')
TOP_TERRITORIOS=por_territorio.head(NUMERO_TERRITORIOS_DETALHE).unidade.tolist()
display(conferencia);display(por_territorio.head(20))
print('territórios para mapas individuais:',por_territorio.head(NUMERO_TERRITORIOS_DETALHE)[['nome','grupo','registros']].to_dict('records'))

# Categoria do produtor é atributo de cada linha, não uma modalidade da UC.
import unicodedata

def categoria_produtor(v):
    s=str(v).strip()
    chave=unicodedata.normalize('NFKD',s).encode('ascii','ignore').decode().casefold()
    if 'pronaf' in chave or chave in ('pequeno produtor','pequenos produtores'):return 'Pronaf'
    if 'pronamp' in chave or chave in ('medio produtor','medios produtores'):return 'Pronamp'
    if 'demais' in chave or chave in ('grande produtor','grandes produtores'):return 'Demais Produtores'
    if chave in ('','nan','none','null','nao informado'):return 'Não informado'
    return 'Não classificado: '+s

x['categoria_produtor']=x.produtor_original.map(categoria_produtor)
auditoria_produtor=x[['produtor_original','categoria_produtor']].drop_duplicates().sort_values('produtor_original')
csv(auditoria_produtor,'auditoria_padronizacao_categoria_produtor.csv')
por_categoria=pd.DataFrame([{'categoria_produtor':cat,'territorios':z.unidade.nunique(),**metricas(z)}
    for cat,z in x.groupby('categoria_produtor')]).sort_values('registros',ascending=False)
csv(por_categoria,'totais_por_categoria_produtor.csv')
por_ano=pd.DataFrame([{'ano':ano,'territorios':z.unidade.nunique(),**metricas(z)} for ano,z in x.groupby('ano')]).sort_values('ano')
csv(por_ano,'totais_por_ano.csv')
def agrupar(campo):
    return pd.DataFrame([{campo:chave,'territorios':z.unidade.nunique(),**metricas(z)}
        for chave,z in x.groupby(campo,dropna=False)]).sort_values('registros',ascending=False)
por_grupo=agrupar('grupo');por_categoria_uc=agrupar('categoria_uc');por_esfera=agrupar('esfera')
for tabela,arquivo in [(por_grupo,'totais_por_grupo_protecao.csv'),
    (por_categoria_uc,'totais_por_categoria_manejo.csv'),(por_esfera,'totais_por_esfera_administrativa.csv')]:csv(tabela,arquivo)
por_uc_produtor=pd.DataFrame([{'unidade':ident,'categoria_produtor':cat,**metricas(z)}
    for (ident,cat),z in tqdm(x.groupby(['unidade','categoria_produtor']),desc='UC × produtor')])
por_uc_produtor=por_uc_produtor.merge(por_territorio[['unidade','nome']],on='unidade',how='left')
csv(por_uc_produtor,'totais_por_uc_e_categoria_produtor.csv')
por_ano_grupo=pd.DataFrame([{'ano':ano,'grupo':grupo,'territorios':z.unidade.nunique(),**metricas(z)}
    for (ano,grupo),z in x.groupby(['ano','grupo'])])
csv(por_ano_grupo,'totais_por_ano_e_grupo.csv')
# Interseção das uniões dos grupos: área compartilhada, sem dupla contagem.
unioes_grupo={g:z.geometry.union_all() for g,z in x.groupby('grupo')}
from itertools import combinations
sobreposicoes=pd.DataFrame([{'grupo_1':a,'grupo_2':b,
    'area_compartilhada_ha':unioes_grupo[a].intersection(unioes_grupo[b]).area/10000}
    for a,b in combinations(unioes_grupo,2)],columns=['grupo_1','grupo_2','area_compartilhada_ha'])
csv(sobreposicoes,'areas_compartilhadas_entre_grupos.csv')
# Se houver IDs repetidos em um resumo, não somar medidas não aditivas.
controle_unidades=pd.DataFrame();resumo_antigo=pd.DataFrame()
if ARQUIVO_RESUMO_UNIDADES.is_file():
    anterior=pd.read_csv(ARQUIVO_RESUMO_UNIDADES,sep=';',encoding='utf-8-sig',dtype=str,keep_default_na=False)
    conv={'_UNIDADE_ID':'unidade','_UNIDADE_NOME':'nome_v2','QUANTIDADE_REGISTROS':'registros_v2',
          'QUANTIDADE_GLEBAS_DISTINTAS':'glebas_v2','AREA_ACUMULADA_HA':'area_acumulada_v2',
          'AREA_FISICA_UNICA_HA':'area_uniao_v2'}
    if set(conv)-set(anterior.columns):raise ValueError('Cabeçalho diferente no resumo V2 por unidade.')
    resumo_antigo=anterior.copy();csv(resumo_antigo,'resumo_v2_original_unidades.csv')
    anterior=anterior[list(conv)].rename(columns=conv);anterior.unidade=anterior.unidade.str.strip()
    for c in ['registros_v2','glebas_v2','area_acumulada_v2','area_uniao_v2']:anterior[c]=anterior[c].map(numero_pt)
    anterior=anterior.groupby('unidade',as_index=False).agg(nome_v2=('nome_v2','first'),
        linhas_resumo_v2=('unidade','size'),registros_v2=('registros_v2','sum'),
        area_acumulada_v2=('area_acumulada_v2','sum'),
        glebas_v2_se_linha_unica=('glebas_v2','first'),uniao_v2_se_linha_unica=('area_uniao_v2','first'))
    multi=anterior.linhas_resumo_v2.gt(1)
    anterior.loc[multi,['glebas_v2_se_linha_unica','uniao_v2_se_linha_unica']]=np.nan
    controle_unidades=por_territorio[['unidade','nome','registros','glebas','area_acumulada_ha','area_uniao_ha']].merge(anterior,on='unidade',how='outer',indicator=True)
    controle_unidades['confere_registros']=controle_unidades.registros.eq(controle_unidades.registros_v2)
    controle_unidades['confere_area_acumulada']=(controle_unidades.area_acumulada_ha-controle_unidades.area_acumulada_v2).abs().le(TOLERANCIA_AREA_HA)
    unicas=controle_unidades.linhas_resumo_v2.eq(1)
    controle_unidades['confere_glebas_linha_unica']=controle_unidades.glebas.eq(controle_unidades.glebas_v2_se_linha_unica).astype('boolean').where(unicas,pd.NA)
    controle_unidades['confere_uniao_linha_unica']=(controle_unidades.area_uniao_ha-controle_unidades.uniao_v2_se_linha_unica).abs().le(TOLERANCIA_AREA_HA).astype('boolean').where(unicas,pd.NA)
    controle_unidades['confere']=(controle_unidades['_merge'].eq('both') & controle_unidades.confere_registros &
        controle_unidades.confere_area_acumulada & controle_unidades.confere_glebas_linha_unica.fillna(True) &
        controle_unidades.confere_uniao_linha_unica.fillna(True))
    controle_unidades['escopo_conferencia']=np.where(unicas,'Registros, glebas, área acumulada e união',
        'Registros e área acumulada; glebas e união recalculadas, sem referência agregável')
    csv(controle_unidades,'conferencia_todas_unidades_com_resumo_v2.csv')
    ORIGENS['resumo_unidades']={'arquivo':str(ARQUIVO_RESUMO_UNIDADES),'sha256':hash_arquivo(ARQUIVO_RESUMO_UNIDADES)}
    print('Resumo V2:',len(resumo_antigo),'linhas;',len(anterior),'IDs; métricas comparáveis conferem em',int(controle_unidades.confere.sum()),'unidades.')
    if not controle_unidades.confere.all():print('ATENÇÃO: há diferenças nas métricas comparáveis; consulte a conferência por unidade.')
else:print('Resumo V2 não encontrado. Os totais atuais foram calculados diretamente nos registros.')

controle_produtor=pd.DataFrame()
if ARQUIVO_RESUMO_PRODUTOR.is_file():
    rp=pd.read_csv(ARQUIVO_RESUMO_PRODUTOR,sep=';',encoding='utf-8-sig',dtype=str,keep_default_na=False)
    col={'_TIPO_PRODUTOR':'rotulo_v2','QUANTIDADE_REGISTROS':'registros_v2',
        'QUANTIDADE_REF_BACEN_DISTINTOS':'refs_v2','VALOR_TOTAL_LINHAS_R':'valor_v2'}
    if set(col)-set(rp.columns):raise ValueError('Cabeçalho diferente no resumo por produtor.')
    rp=rp[list(col)].rename(columns=col);rp['categoria_produtor']=rp.rotulo_v2.map(categoria_produtor)
    if rp.categoria_produtor.duplicated().any():raise ValueError('Categorias repetidas no resumo de produtor; não somar referências distintas.')
    for c in ['registros_v2','refs_v2','valor_v2']:rp[c]=rp[c].map(numero_pt)
    controle_produtor=por_categoria.merge(rp,on='categoria_produtor',how='outer')
    controle_produtor['confere']=(controle_produtor.registros.eq(controle_produtor.registros_v2) &
        controle_produtor.refs.eq(controle_produtor.refs_v2) &
        (controle_produtor.soma_valores_linhas-controle_produtor.valor_v2).abs().le(.02))
    csv(controle_produtor,'conferencia_categoria_produtor_com_v2.csv')
    if not controle_produtor.confere.all():print('ATENÇÃO: diferenças na tabela por categoria de produtor.')
display(por_categoria)

# Confere cada análise mantida contra o respectivo resumo V2.
conferencias_analises={}
def conferir_resumo(atual,campo,caminho,coluna_chave,financeiro=False):
    if not caminho.is_file():
        print('Referência não encontrada (análise atual mantida):',caminho)
        return pd.DataFrame()
    ref=pd.read_csv(caminho,sep=';',dtype=str,keep_default_na=False)
    mapa={'QUANTIDADE_REGISTROS':'registros','QUANTIDADE_GLEBAS_DISTINTAS':'glebas',
      'QUANTIDADE_UNIDADES_DISTINTAS':'territorios','AREA_ACUMULADA_HA':'area_acumulada_ha',
      'AREA_FISICA_UNICA_HA':'area_uniao_ha','QUANTIDADE_REF_BACEN_DISTINTOS':'refs','VALOR_TOTAL_LINHAS_R':'soma_valores_linhas'}
    mapa={k:v for k,v in mapa.items() if k in ref.columns and v in atual.columns}
    if coluna_chave not in ref:raise ValueError('Resumo com chave inesperada: '+str(caminho))
    ref=ref[[coluna_chave,*mapa]].rename(columns={coluna_chave:campo,**{k:v+'_v2' for k,v in mapa.items()}})
    ref[campo]=ref[campo].str.strip()
    if ref[campo].duplicated().any():raise ValueError('Chaves repetidas no resumo: '+str(caminho))
    for v in mapa.values():ref[v+'_v2']=ref[v+'_v2'].map(numero_pt)
    atual=atual.copy();atual[campo]=atual[campo].astype(str)
    out=atual.merge(ref,on=campo,how='outer',indicator=True);out['confere']=out['_merge'].eq('both')
    for v in mapa.values():
        tol=TOLERANCIA_AREA_HA if v.startswith('area') else (.02 if v=='soma_valores_linhas' else 0)
        out['diferenca_'+v]=out[v]-out[v+'_v2']
        out['confere_'+v]=out['diferenca_'+v].abs().le(tol)
        out['confere'] &= out['confere_'+v]
    csv(out,'conferencia_'+campo+'_com_v2.csv')
    ORIGENS['resumo_'+campo]={'arquivo':str(caminho),'sha256':hash_arquivo(caminho)}
    if not out.confere.all():print('ATENÇÃO: diferenças na análise por',campo)
    return out
for atual,campo,caminho,chave in [(por_grupo,'grupo',ARQUIVO_RESUMO_GRUPOS,'_TIPOLOGIA_2'),
 (por_categoria_uc,'categoria_uc',ARQUIVO_RESUMO_CATEGORIAS,'_TIPOLOGIA_1'),
 (por_esfera,'esfera',ARQUIVO_RESUMO_ESFERAS,'_TIPOLOGIA_3'),
 (por_ano,'ano',ARQUIVO_RESUMO_ANOS,'_ANO')]:
    conferencias_analises[campo]=conferir_resumo(atual,campo,caminho,chave)
display(por_grupo);display(por_categoria_uc);display(por_ano)


## 6. Selecionar a UF pelo número de registros
Cada linha original conta uma vez em cada UF com área positiva de interseção, independentemente do número de fragmentos. Uma linha que cruza uma divisa pode estar presente em mais de uma UF. A UF do atributo da UC é mantida nas tabelas; a escolha do mapa usa a localização geométrica. Os valores do resumo estadual anterior são usados apenas para conferência.

In [ ]:
partes=[]
for _,u in ufs.iterrows():
    candidatos=x.iloc[x.sindex.query(u.geometry,predicate='intersects')].copy()
    if candidatos.empty:continue
    candidatos.geometry=candidatos.geometry.intersection(u.geometry)
    candidatos=candidatos.loc[~candidatos.geometry.is_empty & candidatos.geometry.area.gt(0)].copy()
    if candidatos.empty:continue
    candidatos['UF']=u.UF;candidatos['area_ha']=candidatos.geometry.area/10000
    partes.append(candidatos)
if not partes:raise ValueError('Nenhuma interseção localizada nas UFs. Confira os CRS.')
xuf=gpd.GeoDataFrame(pd.concat(partes,ignore_index=True),crs=CRS_ANALISE)
if set(xuf.linha)!=set(x.linha):raise ValueError('Há registros não alcançados pela malha de UFs; confira os limites.')
if xuf.duplicated(['UF','linha']).any():raise AssertionError('Mais de uma parte por registro/UF no ranking.')
por_uf=[]
for uf,z in xuf.groupby('UF'):
    por_uf.append({'UF':uf,'registros':z.linha.nunique(),'glebas':z.gleba.nunique(),'territorios':z.unidade.nunique(),
                   'area_acumulada_ha':z.area_ha.sum(),'area_uniao_ha':z.geometry.union_all().area/10000})
por_uf=pd.DataFrame(por_uf).sort_values(['registros','UF'],ascending=[False,True]).reset_index(drop=True)
UF_TOP=por_uf.iloc[0].UF
csv(por_uf,'ranking_ufs_por_registros.csv')
csv(xuf[['linha','UF','unidade','gleba','ref','area_ha']],'vinculo_registro_uf.csv')
residuo=abs(float(xuf.area_ha.sum())-geral['area_acumulada_ha'])
if residuo>TOLERANCIA_AREA_HA:print(f'Atenção: diferença de {residuo:.6f} ha no corte estadual; confira os limites de contexto.')
display(por_uf)
print('UF selecionada por quantidade de registros:',UF_TOP)

# Confronta UF textual do território com a localização efetiva das interseções.
auditoria_uf=xuf[['unidade','nome','uf_territorio','UF']].drop_duplicates().copy()
NOMES_UFS={'ACRE':'AC','ALAGOAS':'AL','AMAPA':'AP','AMAZONAS':'AM','BAHIA':'BA','CEARA':'CE',
 'DISTRITO FEDERAL':'DF','ESPIRITO SANTO':'ES','GOIAS':'GO','MARANHAO':'MA','MATO GROSSO':'MT',
 'MATO GROSSO DO SUL':'MS','MINAS GERAIS':'MG','PARA':'PA','PARAIBA':'PB','PARANA':'PR','PERNAMBUCO':'PE',
 'PIAUI':'PI','RIO DE JANEIRO':'RJ','RIO GRANDE DO NORTE':'RN','RIO GRANDE DO SUL':'RS','RONDONIA':'RO',
 'RORAIMA':'RR','SANTA CATARINA':'SC','SAO PAULO':'SP','SERGIPE':'SE','TOCANTINS':'TO'}
def siglas_uf(txt):
    t=unicodedata.normalize('NFKD',str(txt)).encode('ascii','ignore').decode().upper().strip()
    partes=re.split(r'[,;/|]+',t)
    return sorted(set(NOMES_UFS.get(p.strip(),p.strip()) for p in partes if p.strip()))
auditoria_uf['siglas_normalizadas']=auditoria_uf.uf_territorio.map(lambda v:','.join(siglas_uf(v)))
auditoria_uf['uf_informada_inclui_uf_espacial']=[uf in siglas_uf(txt) for uf,txt in zip(auditoria_uf.UF,auditoria_uf.uf_territorio)]
csv(auditoria_uf,'auditoria_uf_atributo_versus_espacial.csv')
if not auditoria_uf.uf_informada_inclui_uf_espacial.all():
    print('Há diferenças entre a UF do atributo e a UF das interseções. Os mapas usam a localização geométrica.')
    display(auditoria_uf.loc[~auditoria_uf.uf_informada_inclui_uf_espacial])

# A soma por UF pode ultrapassar o total de linhas quando uma ocorrência cruza uma divisa.
multiplas=xuf.groupby('linha').UF.nunique()
linhas_multiuf=multiplas.loc[multiplas.gt(1)].index
csv(xuf.loc[xuf.linha.isin(linhas_multiuf),['linha','unidade','nome','UF','area_ha']],'registros_em_mais_de_uma_uf.csv')
referencia_uf=pd.DataFrame({'UF':['MS','TO','BA'],'uniao_ha_texto_anterior':[235658.62,211234.66,128249.05]})
conferencia_uf=referencia_uf.merge(por_uf[['UF','area_uniao_ha']],on='UF',how='left')
conferencia_uf['diferenca_ha']=conferencia_uf.area_uniao_ha-conferencia_uf.uniao_ha_texto_anterior
csv(conferencia_uf,'conferencia_areas_uf_com_texto_anterior.csv')
print('Registros em mais de uma UF:',len(linhas_multiuf))
print('Áreas estaduais: recálculo direto com a malha IBGE 2024; diferenças com o resumo anterior ficam registradas.')
display(conferencia_uf)

controle_rank_uf=pd.DataFrame()
if ARQUIVO_RESUMO_UFS.is_file():
    ant=pd.read_csv(ARQUIVO_RESUMO_UFS,sep=';',decimal=',')
    controle_rank_uf=ant[['UF','QUANTIDADE_REGISTROS_ORIGINAIS','AREA_FISICA_UNICA_HA']].rename(columns={
      'QUANTIDADE_REGISTROS_ORIGINAIS':'registros_resumo_v2','AREA_FISICA_UNICA_HA':'uniao_resumo_v2'})
    controle_rank_uf=controle_rank_uf.merge(por_uf[['UF','registros','area_uniao_ha']],on='UF',how='outer')
    controle_rank_uf['diferenca_registros']=controle_rank_uf.registros-controle_rank_uf.registros_resumo_v2
    controle_rank_uf['diferenca_uniao_ha']=controle_rank_uf.area_uniao_ha-controle_rank_uf.uniao_resumo_v2
    csv(controle_rank_uf,'conferencia_ufs_atual_versus_v2.csv')
    ORIGENS['resumo_ufs']={'arquivo':str(ARQUIVO_RESUMO_UFS),'sha256':hash_arquivo(ARQUIVO_RESUMO_UFS)}


## 7. Mapas com limites visíveis e ampliações
Quatro mapas: Cerrado; UF com mais registros e ampliação; duas UCs com mais registros. Azul forte = limite da UC; laranja = gleba completa; vermelho = interseção.

Cada mapa individual apresenta a UC completa e uma ampliação das glebas associadas à célula de uma grade 4 × 4 que contém mais pontos internos das linhas de interseção. Esse critério serve somente ao enquadramento cartográfico, não altera os rankings nem classifica irregularidades. As geometrias completas são usadas no detalhe; a simplificação é exclusiva das visões geral e estadual.

In [ ]:
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'axes.titleweight':'bold','savefig.facecolor':'white','svg.fonttype':'none'})
COR_UC='#164D87';COR_GLEBA='#C67500';COR_INTER='#C52845';COR_CERRADO='#E9EFDE';COR_UF='#575F65'
mapas_gerados=[]
para_ll=Transformer.from_crs(CRS_ANALISE,4674,always_xy=True)
para_xy=Transformer.from_crs(4674,CRS_ANALISE,always_xy=True)
geod=Geod(ellps='GRS80')

def serie(geom):return gpd.GeoSeries([geom],crs=CRS_ANALISE)
def enquadrar(ax,bounds,margem=0.07):
    a,b,c,d=map(float,bounds);w=max(c-a,1000);h=max(d-b,1000)
    ax.set_xlim(a-margem*w,c+margem*w);ax.set_ylim(b-margem*h,d+margem*h)
    ax.set_aspect('equal');ax.set_axis_off()

def perto(z,bounds):
    ids=z.sindex.query(box(*bounds),predicate='intersects')
    return z.iloc[ids]

def adornos(ax):
    a,c=ax.get_xlim();b,d=ax.get_ylim();w=c-a;h=d-b
    # Barra horizontal com comprimento calibrado geodesicamente no local.
    x0=a+0.055*w;y0=b+0.055*h
    lo,la=para_ll.transform(x0,y0);lo2,la2=para_ll.transform(x0+0.23*w,y0)
    _,_,dist=geod.inv(lo,la,lo2,la2)
    exp=10**math.floor(math.log10(max(dist,1)))
    length=max(v for v in [1*exp,2*exp,5*exp,10*exp,0.5*exp] if v<=dist)
    dx=0.23*w*length/dist
    ax.plot([x0,x0+dx],[y0,y0],color='white',lw=7,zorder=30)
    ax.plot([x0,x0+dx/2],[y0,y0],color='#20262B',lw=4,zorder=31)
    ax.plot([x0+dx/2,x0+dx],[y0,y0],color='#8C969C',lw=4,zorder=31)
    txt=f'{length/1000:g} km' if length>=1000 else f'{length:g} m'
    ax.text(x0,y0+0.016*h,'0',fontsize=8,zorder=32,ha='center',bbox=dict(fc='white',ec='none',pad=1))
    ax.text(x0+dx,y0+0.016*h,txt,fontsize=8,zorder=32,ha='center',bbox=dict(fc='white',ec='none',pad=1))
    # Norte verdadeiro no ponto de implantação da seta.
    nx=a+0.95*w;ny=b+0.85*h;lon,lat=para_ll.transform(nx,ny)
    xx,yy=para_xy.transform(lon,lat+0.1);vx=xx-nx;vy=yy-ny;norm=math.hypot(vx,vy)
    ex=nx+vx/norm*h*0.08;ey=ny+vy/norm*h*0.08
    ax.annotate('',xy=(ex,ey),xytext=(nx,ny),arrowprops=dict(arrowstyle='-|>',color='#20262B',lw=1.4),zorder=35)
    ax.text(ex,ey+0.015*h,'N',ha='center',va='bottom',weight='bold',fontsize=10,zorder=35,bbox=dict(fc='white',ec='none',pad=1))

def desenhar(ax,territorio_ids,gleba_ids,ocorrencias,bounds,escala='geral',estado=None):
    a,b,c,d=bounds;w=max(c-a,1000);h=max(d-b,1000)
    janela=box(a-.10*w,b-.10*h,c+.10*w,d+.10*h)
    bg=perto(ufs,janela.bounds).copy()
    bg.geometry=bg.geometry.intersection(janela)
    bg.plot(ax=ax,facecolor='#FAFBFC',edgecolor=COR_UF,linewidth=0.55,zorder=0)
    contexto=cerrado.intersection(janela)
    if not contexto.is_empty:serie(contexto).plot(ax=ax,facecolor=COR_CERRADO,edgecolor='#768C65',linewidth=0.65,zorder=1)
    if escala=='geral':
        ts=territorios.loc[territorios.unidade.isin(territorio_ids)];gs=glebas.loc[glebas.gleba.isin(gleba_ids)]
    else:
        tolerancia=SIMPLIFICACAO_ESTADO_M if escala=='estado' else 0
        ts=carregar_geometrias('territorio',territorio_ids,tolerancia)
        gs=carregar_geometrias('gleba',gleba_ids,tolerancia)
    # Cópias sobrepostas desenhadas uma vez; as contagens continuam usando todas as linhas.
    ocorrencias=ocorrencias[['geometry']].drop_duplicates('geometry').copy()
    if escala in ['geral','estado']:
        tol=SIMPLIFICACAO_GERAL_M if escala=='geral' else SIMPLIFICACAO_ESTADO_M
        ocorrencias.geometry=ocorrencias.geometry.simplify(tol,preserve_topology=True)
    if not ts.empty:ts.plot(ax=ax,facecolor='#DDE9F3',edgecolor=COR_UC,linewidth=1.5 if escala=='geral' else 2.0,alpha=0.95,zorder=3)
    if not ocorrencias.empty:ocorrencias.plot(ax=ax,facecolor=COR_INTER,edgecolor='#8A1530',linewidth=0.35,alpha=0.90,zorder=5,rasterized=True)
    if not gs.empty:gs.boundary.plot(ax=ax,color=COR_GLEBA,linewidth=0.65 if escala=='geral' else 0.9,zorder=6,rasterized=True)
    # Redesenha o contorno do território sobre as interseções.
    if not ts.empty:ts.boundary.plot(ax=ax,color=COR_UC,linewidth=1.2 if escala=='geral' else 1.65,zorder=7)
    if estado is not None:serie(estado).boundary.plot(ax=ax,color='#252C31',linewidth=1.4,zorder=7)
    enquadrar(ax,bounds);adornos(ax)

handles=[Patch(facecolor=COR_CERRADO,edgecolor='#768C65',label='Bioma Cerrado'),
         Patch(facecolor='#DDE9F3',edgecolor=COR_UC,linewidth=1.8,label='Unidade de conservação com ocorrência'),
         Patch(facecolor='none',edgecolor=COR_GLEBA,linewidth=1.2,label='Limite da gleba'),
         Patch(facecolor=COR_INTER,edgecolor='#8A1530',label='Área de interseção'),
         Patch(facecolor='none',edgecolor=COR_UF,linewidth=1,label='Limite de UF')]

def finalizar(fig,nome,titulo,subtitulo,nota=''):
    fig.suptitle(titulo,x=0.06,y=0.965,ha='left',fontsize=16,weight='bold',color='#202C36')
    fig.text(0.06,0.924 if '\n' not in titulo else 0.89,subtitulo,ha='left',fontsize=10,color='#45535D')
    fig.legend(handles=handles,loc='lower center',bbox_to_anchor=(0.5,0.068),ncol=3,frameon=False,fontsize=9)
    fonte='Fontes: SICOR/BCB e MMA/CNUC, geometrias preservadas na base V2; IBGE, biomas 2019 e UFs 2024.'
    if ARQUIVO_UFS or ARQUIVO_BIOMAS:fonte='Fontes: SICOR/BCB e MMA/CNUC (base V2); '+FONTE_BIOMAS+'; '+FONTE_UFS+'.'
    fig.text(0.06,0.047,fonte,fontsize=7.5,color='#45535D')
    fig.text(0.06,0.031,'SIRGAS 2000 / Brazil Polyconic (EPSG:5880). '+nota,fontsize=7.5,color='#45535D')
    fig.subplots_adjust(left=0.04,right=0.97,bottom=0.15,top=0.89 if '\n' not in titulo else 0.845)
    for ext in FORMATOS:fig.savefig(SAIDA/'mapas'/f'{nome}.{ext}',dpi=DPI)
    mapas_gerados.append({'arquivo':nome,'titulo':titulo,'subtitulo':subtitulo,'nota':nota})
    if MOSTRAR_FIGURAS:plt.show()
    plt.close(fig)

fig,ax=plt.subplots(figsize=(12,12))
desenhar(ax,territorios.unidade,glebas.gleba,x,cerrado.bounds)
for _,u in ufs.loc[ufs.intersects(cerrado)].iterrows():
    pt=u.geometry.intersection(cerrado).representative_point()
    ax.text(pt.x,pt.y,u.UF,fontsize=9,weight='bold',color='#323B41',ha='center',zorder=20,
            bbox=dict(fc='white',ec='none',alpha=0.65,pad=1))
finalizar(fig,'01_cerrado_ucs_contornos_reforcados','Interseções com unidades de conservação no Cerrado',
          f"2021–2025 · {len(x):,} registros · {len(glebas):,} glebas distintas · {len(territorios)} unidades territoriais".replace(',','.'),
          f'Desenho simplificado em {SIMPLIFICACAO_GERAL_M} m; ampliações nos mapas de detalhe.')

# Estado completo à esquerda e ampliação do território com mais registros nesse estado à direita.
z=xuf.loc[xuf.UF.eq(UF_TOP)].copy();estado=ufs.loc[ufs.UF.eq(UF_TOP)].geometry.iloc[0]
rank_local=z.groupby('unidade').linha.nunique().rename('n').reset_index().merge(territorios[['unidade','nome']],on='unidade')
top_local=rank_local.sort_values(['n','nome','unidade'],ascending=[False,True,True]).iloc[0].unidade
zl=z.loc[z.unidade.eq(top_local)];territorio_local=carregar_geometrias('territorio',[top_local]).iloc[0]
gl_local=carregar_geometrias('gleba',zl.gleba)
bdet=gpd.GeoSeries([territorio_local.geometry,*gl_local.geometry],crs=CRS_ANALISE).total_bounds
fig,axs=plt.subplots(1,2,figsize=(15,10),gridspec_kw={'width_ratios':[1,1.15]})
desenhar(axs[0],z.unidade.unique(),z.gleba.unique(),z,estado.bounds,'estado',estado)
desenhar(axs[1],[top_local],zl.gleba.unique(),zl,bdet,'detalhe',estado)
axs[0].set_title(f'{UF_TOP} · limite estadual completo',fontsize=10,pad=8)
axs[1].set_title('Ampliação: '+'\n'.join(textwrap.wrap(territorio_local['nome'],width=44)),fontsize=10,pad=8)
a,b,c,e=bdet;axs[0].add_patch(Rectangle((a,b),c-a,e-b,fill=False,edgecolor='#5B296F',linewidth=1.4,linestyle='--',zorder=25))
finalizar(fig,'02_uf_mais_casos_com_ampliacao',f'{UF_TOP} · UF com mais registros de interseção',
          f'{z.linha.nunique()} registros · {z.gleba.nunique()} glebas distintas · {z.unidade.nunique()} unidades territoriais · 2021–2025',
          f'Tracejado: ampliação. Desenho estadual simplificado em {SIMPLIFICACAO_ESTADO_M} m; detalhe integral.')

ampliacoes=[]
def janela_detalhe(z):
    # Grade regular 4 × 4 sobre a extensão das interseções desta UC.
    a,b,c,d=z.total_bounds;w=max(c-a,100);h=max(d-b,100)
    pontos=z.geometry.representative_point()
    ix=np.minimum(3,np.maximum(0,((pontos.x-a)/w*4).astype(int)))
    iy=np.minimum(3,np.maximum(0,((pontos.y-b)/h*4).astype(int)))
    counts=pd.DataFrame({'ix':ix,'iy':iy}).value_counts().rename('n').reset_index()
    best=counts.sort_values(['n','iy','ix'],ascending=[False,True,True]).iloc[0]
    selecionadas=z.loc[(ix==best.ix)&(iy==best.iy)]
    # Inclui os limites completos das glebas selecionadas, evitando cortar uma gleba
    # grande em uma célula muito pequena. Janela quadrada favorece a leitura do detalhe.
    gs=carregar_geometrias('gleba',selecionadas.gleba)
    xa,ya,xb,yb=gs.total_bounds;lado=max(xb-xa,yb-ya,1000)
    cx=(xa+xb)/2;cy=(ya+yb)/2
    return (cx-lado/2,cy-lado/2,cx+lado/2,cy+lado/2),int(best.n)

for pos,ident in enumerate(TOP_TERRITORIOS,1):
    t=carregar_geometrias('territorio',[ident]).iloc[0];zi=x.loc[x.unidade.eq(ident)]
    gi=carregar_geometrias('gleba',zi.gleba)
    limites=gpd.GeoSeries([t.geometry,*gi.geometry],crs=CRS_ANALISE).total_bounds
    zoom,n_pontos=janela_detalhe(zi)
    fig,axs=plt.subplots(1,2,figsize=(15,10),gridspec_kw={'width_ratios':[1,1.2]})
    desenhar(axs[0],[ident],gi.gleba,zi,limites,'detalhe')
    desenhar(axs[1],[ident],gi.gleba,zi,zoom,'detalhe')
    for ax,lim in zip(axs,[limites,zoom]):
        ub=perto(ufs,lim).boundary.intersection(box(*lim));ub=ub.loc[~ub.is_empty]
        if not ub.empty:ub.plot(ax=ax,color='#252C31',linewidth=.8,zorder=8)
    a,b,c,e=zoom;axs[0].add_patch(Rectangle((a,b),c-a,e-b,fill=False,edgecolor='#5B296F',lw=1.4,linestyle='--',zorder=25))
    axs[0].set_title('Limite completo da UC e glebas',fontsize=10,pad=8)
    axs[1].set_title('Ampliação da concentração local',fontsize=10,pad=8)
    m=por_territorio.loc[por_territorio.unidade.eq(ident)].iloc[0]
    titulo='\n'.join(textwrap.wrap(t['nome'],width=72))
    finalizar(fig,f'{pos+2:02d}_uc_{re.sub(r"[^A-Za-z0-9_-]","_",str(ident))}_detalhe',titulo,
       f"ID {ident} · {t.grupo} · {m.registros} registros · {m.glebas} glebas distintas · 2021–2025",
       'Tracejado: ampliação. Glebas e interseções distintas; geometrias de detalhe sem simplificação.')
    ampliacoes.append({'unidade':ident,'bounds_epsg_5880':list(map(float,zoom)),
      'pontos_na_celula_selecionada':n_pontos,'criterio':'Glebas da célula 4 × 4 com mais pontos internos de registros; apenas desenho'})


## 8. Tabelas, gráficos e textos para o subcapítulo
Exporta CSV, Excel e HTML com a relação completa por UC e as análises mantidas. Os gráficos mostram registros por grupo e por ano, separando frequência de soma financeira. O texto de apoio apresenta os valores efetivamente calculados e sinaliza diferenças nas conferências.

In [ ]:
def br(v,casas=0):
    if pd.isna(v):return 'Não informado'
    return f'{float(v):,.{casas}f}'.replace(',','X').replace('.',',').replace('X','.')
nota=('Fonte: elaborado pelo autor com base nas interseções SICOR × CNUC, versão 2. '
      'Cada registro corresponde a uma linha original. A UC é identificada por UC_ID_INTERNO. '
      'Glebas e referências distintas e união geométrica são recalculadas, não somadas entre UCs. '
      'Valores financeiros das linhas podem repetir valores administrativos e não são rateados pela área. '
      'UF informada = atributo UC_UF; ranking estadual = recorte direto das interseções. Áreas em EPSG:5880.')
def tabela_html(t,titulo,arquivo,nota_extra=''):
    from html import escape
    s=('<!doctype html><html lang="pt-BR"><meta charset="utf-8"><title>'+escape(titulo)+
       '</title><style>body{font:12px Arial;margin:24px}table{border-collapse:collapse;width:100%}'
       'th,td{padding:6px;border:1px solid #bbb}th{background:#e4edf4}p{line-height:1.5}'
       '@media print{thead{display:table-header-group}tr{break-inside:avoid}@page{size:A4 landscape;margin:15mm}}</style>'
       '<h2>'+escape(titulo)+'</h2>'+t.to_html(index=False,escape=True,border=0)+'<p>'+escape(nota+' '+nota_extra)+'</p></html>')
    (SAIDA/'tabelas'/arquivo).write_text(s,encoding='utf-8')

def formatar(t):
    t=t.copy()
    for c in ['registros','glebas','refs','territorios']:
        if c in t:t[c]=t[c].map(br)
    for c in [c for c in t if c.startswith('area_')]:t[c]=t[c].map(lambda v:br(v,4))
    if 'soma_valores_linhas' in t:t['soma_valores_linhas']=t.soma_valores_linhas.map(lambda v:br(v,2))
    return t.rename(columns={'unidade':'ID interno','nome':'Unidade de conservação','grupo':'Grupo de proteção',
      'categoria_uc':'Categoria de manejo','esfera':'Esfera administrativa','uf_territorio':'UF informada',
      'registros':'Registros','glebas':'Glebas distintas','refs':'REF_BACEN distintos','territorios':'UCs distintas',
      'area_acumulada_ha':'Área acumulada (ha)','area_uniao_ha':'União geométrica (ha)',
      'soma_valores_linhas':'Valor somado nas linhas (R$)','ano':'Ano','categoria_produtor':'Categoria de produtor'})
cols=['unidade','nome','grupo','categoria_uc','uf_territorio','registros','glebas','refs','area_acumulada_ha','area_uniao_ha']
tab=por_territorio[cols].copy()
total={k:'—' for k in cols};total.update(geral,unidade='TOTAL',nome='Conjunto completo')
tab=pd.concat([tab,pd.DataFrame([{k:total[k] for k in cols}])],ignore_index=True)
tabela_html(formatar(tab),'Totais por unidade de conservação — Cerrado, 2021–2025','tabela_completa_por_uc.html',
 'Áreas com quatro casas decimais para permitir visualizar pequenas interseções; não expressam precisão posicional.')
tabela_html(formatar(por_territorio.head(20)[cols]),'20 UCs com mais registros — Cerrado, 2021–2025','tabela_20_ucs.html','Tabela parcial; consultar a relação completa.')
csv(por_territorio.head(20),'20_ucs_com_mais_registros.csv')
gcols=['grupo','registros','glebas','territorios','area_acumulada_ha','area_uniao_ha']
tg=pd.concat([por_grupo[gcols],pd.DataFrame([{'grupo':'Total geral',**geral}])[gcols]],ignore_index=True)
tabela_html(formatar(tg),'Interseções por grupo de proteção — Cerrado, 2021–2025','tabela_grupos_protecao.html',
 'As uniões por grupo podem compartilhar área; o total geral é a união de todas as interseções.')

# Dois gráficos preservam a distinção entre frequência e materialidade financeira.
graficos_gerados=[]
def salvar_grafico(fig,arquivo,titulo):
    fig.text(.02,.015,'Fonte: SICOR/BCB e CNUC, V2. Registros de interseção; não são operações únicas ou concessões irregulares.',fontsize=8,color='#45535D')
    fig.tight_layout(rect=(0,.06,1,.95));fig.suptitle(titulo,fontsize=16,weight='bold')
    for ext in FORMATOS:fig.savefig(SAIDA/'graficos'/f'{arquivo}.{ext}',dpi=DPI)
    graficos_gerados.append({'arquivo':arquivo,'titulo':titulo})
    if MOSTRAR_FIGURAS:plt.show()
    plt.close(fig)
fig,axs=plt.subplots(1,2,figsize=(14,6));g=por_grupo.iloc[::-1]
for ax,campo,titulo,esc in [(axs[0],'registros','Registros de interseção',1),(axs[1],'area_uniao_ha','Área da união por grupo (mil ha)',1000)]:
    vals=g[campo]/esc;ax.barh(g.grupo,vals,color=['#B44C41' if k=='Proteção Integral' else '#167D8D' for k in g.grupo])
    ax.set_title(titulo,fontsize=11);ax.set_xlim(0,max(vals)*1.28)
    for i,v in enumerate(vals):ax.text(v+max(vals)*.02,i,br(v,0 if esc==1 else 2),va='center',fontsize=10)
    ax.spines[['top','right']].set_visible(False);ax.tick_params(axis='y',labelsize=9)
salvar_grafico(fig,'01_grupos_protecao','Interseções por grupo de proteção — 2021–2025')
fig,axs=plt.subplots(1,2,figsize=(14,6));anos=por_ano.ano.astype(str)
axs[0].bar(anos,por_ano.registros,color='#167D8D');axs[0].set_title('Registros por ano',fontsize=11)
axs[1].bar(anos,por_ano.soma_valores_linhas/1e9,color='#AC7A20');axs[1].set_title('Valor somado nas linhas (R$ bilhões)',fontsize=11)
for ax,vals,casas in [(axs[0],por_ano.registros,0),(axs[1],por_ano.soma_valores_linhas/1e9,2)]:
    ax.set_ylim(0,max(vals)*1.18);ax.spines[['top','right']].set_visible(False)
    for i,v in enumerate(vals):ax.text(i,v+max(vals)*.02,br(v,casas),ha='center',fontsize=10)
salvar_grafico(fig,'02_evolucao_anual','Interseções com UCs e valores nas linhas — 2021–2025')

# Conferência dos destaques presentes no texto anterior (sem forçar resultados).
checar_destaques=[]
def destaque(nome,valor,esperado,tol=0):
    checar_destaques.append({'indicador':nome,'calculado':valor,'referencia':esperado,'confere':bool(abs(valor-esperado)<=tol)})
apa=por_categoria_uc.loc[por_categoria_uc.categoria_uc.eq('Área de Proteção Ambiental')]
if len(apa)==1:
    destaque('Registros em APA',int(apa.iloc[0].registros),17479)
    destaque('União em APA (ha)',float(apa.iloc[0].area_uniao_ha),770185.99,TOLERANCIA_AREA_HA)
for ident,esperado in [('8',2804),('558',1720)]:
    z=por_territorio.loc[por_territorio.unidade.eq(ident)]
    if len(z):destaque('Registros da UC '+ident,int(z.iloc[0].registros),esperado)
controle_destaques=pd.DataFrame(checar_destaques)
csv(controle_destaques,'conferencia_destaques_texto_anterior.csv')

planilhas=[('Por UC',por_territorio),('20 primeiras',por_territorio.head(20)),('Total geral',pd.DataFrame([geral])),
 ('Grupo protecao',por_grupo),('Categoria manejo',por_categoria_uc),('Esfera administrativa',por_esfera),
 ('Por ano',por_ano),('Categoria produtor',por_categoria),('UC x produtor',por_uc_produtor),('Ano x grupo',por_ano_grupo),
 ('Ranking UFs',por_uf),('Area compartilhada grupos',sobreposicoes),('Auditoria UF',auditoria_uf),
 ('Identificacao',auditoria_nomes),('Padronizacao produtor',auditoria_produtor),('Conferencia geral',conferencia),
 ('Conferencia unidades',controle_unidades),('Conferencia produtor',controle_produtor),('Conferencia UFs',controle_rank_uf),
 ('Destaques do texto',controle_destaques),('Reparos desenho',reparos)]
for chave,tb in conferencias_analises.items():planilhas.append(('Conferencia '+chave,tb))
with pd.ExcelWriter(SAIDA/'tabelas'/'Totais_Unidades_Conservacao_5_5_6.xlsx',engine='openpyxl') as writer:
    from openpyxl.styles import Font,PatternFill
    for nome,z in planilhas:
        z.to_excel(writer,sheet_name=nome,index=False);ws=writer.sheets[nome]
        ws.freeze_panes='A2';ws.auto_filter.ref=ws.dimensions
        for cell in ws[1]:cell.font=Font(bold=True,color='FFFFFF');cell.fill=PatternFill('solid',fgColor='164D87')
        for col in ws.columns:
            ws.column_dimensions[col[0].column_letter].width=min(55,max(14,max(len(str(c.value or '')) for c in col)+2))
            if any(v in str(col[0].value) for v in ['area_','soma_valores']):
                for cell in col[1:]:cell.number_format='#,##0.0000' if 'area_' in str(col[0].value) else '#,##0.00'
    pd.DataFrame({'Notas':[nota,'A seleção usa número de registros; os mapas mostram apenas UCs com ocorrência.',
      'Grupo de proteção, categoria de manejo e esfera seguem os atributos da base; não reconstituem a situação jurídica na data da operação.',
      'A interseção não comprova irregularidade. Exige análise documental da categoria, zoneamento, plano de manejo, domínio, datas e regras aplicáveis.',
      'Uniões por grupo e por UC não são aditivas. A soma financeira não representa volume único de operações.',
      'Ampliação cartográfica: glebas da célula de grade 4 × 4 com mais pontos internos das linhas; não é um novo ranking de UCs.',
      'Os mapas de detalhe usam geometrias integrais. Reparos de desenho não substituem as interseções analíticas.'
    ]}).to_excel(writer,sheet_name='Leia-me',index=False)

texto=(f"Foram preservados {br(geral['registros'])} registros, {br(geral['glebas'])} glebas distintas, "
       f"{br(geral['territorios'])} UCs e {br(geral['refs'])} referências BACEN.\n"
       f"Área acumulada: {br(geral['area_acumulada_ha'],2)} ha; união: {br(geral['area_uniao_ha'],2)} ha; "
       f"valores nas linhas: R$ {br(geral['soma_valores_linhas'],2)}.\n\n")
for titulo,tb,chave in [('Grupos de proteção',por_grupo,'grupo'),('Categorias de manejo',por_categoria_uc,'categoria_uc'),
 ('Esferas administrativas',por_esfera,'esfera'),('Anos',por_ano,'ano'),('Categorias de produtor',por_categoria,'categoria_produtor')]:
    texto+=titulo+':\n'
    for _,r in tb.iterrows():
        texto+=f"- {r[chave]}: {br(r.registros)} registros; {br(r.glebas)} glebas; {br(r.refs)} referências; {br(r.area_uniao_ha,2)} ha de união; R$ {br(r.soma_valores_linhas,2)} nas linhas.\n"
    texto+='\n'
texto+=f"UF com mais registros: {UF_TOP}, {br(por_uf.iloc[0].registros)}. Soma estadual: {br(por_uf.registros.sum())}; {len(linhas_multiuf)} linhas em mais de uma UF.\n\n"
texto+='UFs por área física de interseção:\n'
for _,r in por_uf.sort_values('area_uniao_ha',ascending=False).iterrows():texto+=f"- {r.UF}: {br(r.area_uniao_ha,2)} ha; {br(r.registros)} registros.\n"
texto+='\nUCs selecionadas pelo número de registros:\n'
for _,r in por_territorio.head(NUMERO_TERRITORIOS_DETALHE).iterrows():
    texto+=f"- {r['nome']} (ID {r.unidade}; {r.grupo}; {r.categoria_uc}; {r.esfera}; UF informada: {r.uf_territorio}): {br(r.registros)} registros; {br(r.glebas)} glebas; {br(r.refs)} referências; {br(r.area_acumulada_ha,2)} ha acumulados; {br(r.area_uniao_ha,2)} ha de união; R$ {br(r.soma_valores_linhas,2)} nas linhas.\n"
texto+='\nÁrea compartilhada entre uniões de grupos:\n'
for _,r in sobreposicoes.iterrows():texto+=f"- {r.grupo_1} × {r.grupo_2}: {br(r.area_compartilhada_ha,2)} ha.\n"
texto+=(f"\nReparos cartográficos após reprojeção: {len(reparos)}. Nenhuma deduplicação administrativa. "
 'Diferenças estaduais decorrem do recorte direto pela malha IBGE 2024 e estão exportadas. '
 'A UF do atributo territorial foi preservada; nomes completos foram normalizados para siglas apenas na auditoria.\n'
 'Os resultados descrevem interseções, não concessões irregulares. Categoria, zoneamento, plano de manejo, data de criação, domínio, atividade e regras aplicáveis exigem análise documental. '
 'A camada não permite somar suas áreas com as demais camadas sem nova união geométrica.\n')
checagens=[conferencia,controle_unidades,controle_produtor,*conferencias_analises.values()]
if any(not t.empty and not t.confere.all() for t in checagens):
    texto+='\nATENÇÃO: existem divergências com resumos V2. Confira as tabelas antes da reescrita.\n'
(SAIDA/'resultados_para_reescrita_5_5_6.txt').write_text(texto,encoding='utf-8')
legendas=[]
for m in mapas_gerados:
    legendas.append(m['arquivo']+'\n'+m['titulo'].replace('\n',' ')+' — '+m['subtitulo']+'.\n'+m['nota']+
     '\nFonte: elaborado pelo autor com base no SICOR/BCB e no CNUC, limites preservados na V2; '+FONTE_BIOMAS+'; '+FONTE_UFS+'.\n')
for g in graficos_gerados:legendas.append(g['arquivo']+'\n'+g['titulo']+'.\nFonte: elaborado pelo autor com base no SICOR/BCB e no CNUC, versão 2.\n')
(SAIDA/'legendas_das_figuras.txt').write_text('\n'.join(legendas),encoding='utf-8')
print(texto)


## 9. Metadados, ZIP e pasta dos resultados
Compartilhe o ZIP ou a pasta desta execução para a reescrita do subcapítulo. O cache reduzido fica na pasta `_cache`, fora do ZIP. Se a conferência indicar divergências, utilize as tabelas de diagnóstico antes de substituir os números do texto.

In [ ]:
metadados={'execucao_utc':datetime.now(timezone.utc).isoformat(),'origens':ORIGENS,'crs_wkt':CRS_WKT,'crs_calculo':CRS_ANALISE,
 'fontes_contexto':{'ufs':FONTE_UFS,'biomas':FONTE_BIOMAS},'colunas':C,'anos':ANOS,'deduplicacao_analitica':False,
 'unidade_caso':'Linha original da base espacial V2','criterio_selecao':'Quantidade de registros decrescente; desempate por nome/ID ou sigla UF',
 'uf_selecionada':UF_TOP,'territorios_selecionados':TOP_TERRITORIOS,'totais':{k:float(v) if isinstance(v,(np.floating,np.integer)) else v for k,v in geral.items()},
 'contexto_territorios':'Somente UCs com ocorrência; limites da V2. Desenho geral/estadual simplificado, detalhe integral'
 ,'simplificacao_desenho_m':{'geral':SIMPLIFICACAO_GERAL_M,'estado':SIMPLIFICACAO_ESTADO_M,'detalhe':0}
 ,'cache':str(BANCO),'identificacao_base':'caminho, bytes e mtime_ns; não é hash integral','mapas':mapas_gerados,'coluna_nome':C['nome'],'referencia_rank_unidades':{'8':2804,'558':1720},'papel_camada':'Triagem territorial; não comprova irregularidade',
 'reparos_desenho':reparos.to_dict('records'),'categoria_produtor_coluna':C['produtor_original'],
 'grupos_preservados':True,'graficos':graficos_gerados,'ampliacoes':ampliacoes,'divergencias_uf_atributo':int((~auditoria_uf.uf_informada_inclui_uf_espacial).sum()),
 'diferenca_corte_estadual_ha':residuo,'confere_totais_referencia':bool(conferencia.confere.all()),
 'versoes':{'python':platform.python_version(),'pandas':pd.__version__,'geopandas':gpd.__version__,'shapely':shapely.__version__}}
(SAIDA/'metadados.json').write_text(json.dumps(metadados,ensure_ascii=False,indent=2),encoding='utf-8')
zip_final=SAIDA/'Resultados_Unidades_Conservacao_5_5_6.zip'
with zipfile.ZipFile(zip_final,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(SAIDA.rglob('*')):
        if p.is_file() and p!=zip_final:z.write(p,arcname=str(p.relative_to(SAIDA)))
print('Resultados salvos em:',SAIDA)
print('ZIP:',zip_final)
if BAIXAR_ZIP:
    from google.colab import files
    files.download(str(zip_final))


### Produtos e próximos passos
- `tabelas/totais_por_unidade_conservacao_completo.csv`: todas as UCs, uma linha por ID.
- Tabelas por grupo de proteção, categoria de manejo, esfera, ano e produtor; conferências com a V2.
- Quatro mapas e dois gráficos em PNG (400 dpi), PDF e SVG.
- `resultados_para_reescrita_5_5_6.txt`, `legendas_das_figuras.txt` e `metadados.json`.
- `Resultados_Unidades_Conservacao_5_5_6.zip`: pacote para análise e reescrita do Word.

A seleção é por **número de registros**, não por área. As categorias de manejo não são confundidas com categorias de produtor. UCs sem ocorrência na base não integram os mapas. Sobreposição, por si só, não comprova irregularidade. A síntese das seis camadas exige conferência conjunta e não é obtida somando as uniões desta camada com as demais.